<a href="https://colab.research.google.com/github/Jumpr15/pytorch-work/blob/main/rl_REINFORCE_policy_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gymnasium

In [213]:
def loss_fn(Rewards, Log_Probs):
  gamma = 0.95
  loss = 0
  cul_log_prob = 0

  for t in range(torch.tensor(Rewards).size(0)):
    cul_log_prob += Log_Probs[t]
    loss -= Rewards[t] * (gamma ** t) * cul_log_prob

  return loss

def sample_model(state):
  probs = model(state)
  distr = torch.distributions.Categorical(probs=probs)
  action = distr.sample()
  log_prob = (distr.log_prob(action))
  return action.item(), log_prob

def sample_episode(env, state):
  terminated = None
  Actions = []
  States = []
  Rewards = []
  Log_Probs = []

  while not terminated:
    action, log_prob = sample_model(torch.from_numpy(state))
    state, reward, terminated, truncated, info = env.step(action)
    Actions.append(action)
    States.append(state)
    Rewards.append(reward)
    Log_Probs.append(log_prob)

  loss = loss_fn(Rewards, Log_Probs)

  return loss

    # print(action, state, reward, terminated, truncated, info)

In [214]:
torch.tensor(Actions).size(), torch.tensor(States).size(), torch.tensor(Rewards).size(), torch.tensor(Log_Probs).size()

(torch.Size([12]), torch.Size([12, 4]), torch.Size([12]), torch.Size([44]))

In [215]:
cul_log_prob, loss

(tensor(-8.0974, grad_fn=<AddBackward0>),
 tensor(0.6829, grad_fn=<RsubBackward1>))

In [216]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

env = RecordVideo(gym.make("CartPole-v1", render_mode="rgb_array"), video_folder='./video')

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:433: UserWarning: WARN: Unable to save last video! Did you call close()?
  logger.warn("Unable to save last video! Did you call close()?")


In [217]:
import torch.nn as nn
import torch.optim as optim

in_dims = 4
out_dims = 2
hidden_dims = 128

lr = 1e-3

model = nn.Sequential(
    nn.Linear(in_dims, hidden_dims),
    nn.ReLU(),
    nn.Linear(hidden_dims, out_dims),
    nn.Softmax(dim=0)
)

optimizer = optim.Adam(model.parameters(), lr=lr)

In [218]:
import torch

iters = 5

for _ in range(iters):
  state = env.reset() # get initial env state
  state = state[0]

  loss = sample_episode(env, state)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(loss)


tensor(184.5724, grad_fn=<SubBackward0>)
tensor(46.6961, grad_fn=<SubBackward0>)
tensor(33.0530, grad_fn=<SubBackward0>)
tensor(72.0654, grad_fn=<SubBackward0>)
tensor(28.8117, grad_fn=<SubBackward0>)
